# NB-04 · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-04**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-04.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This works through the graph operators behind the price model: incidence, weighted Laplacian, Hodge decomposition, and resistance.

- [Volume II, Chapter 1 — Discrete calculus on graphs](https://sherandoresearch.github.io/power-markets-series/vol2_grid_geometry.html#c1)
- [Volume II, Chapter 3 — Hodge decomposition](https://sherandoresearch.github.io/power-markets-series/vol2_grid_geometry.html#c3)
- [Volume II, §4.2 — The Wheatstone bridge](https://sherandoresearch.github.io/power-markets-series/vol2_grid_geometry.html#c4-w)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-04', require_key=True)
print(setup_status)


# NB-04 — Graph calculus, Hodge decomposition, and resistance
Companion to Volume II §§1–4. Prerequisites: NB-01's incidence and flow signs.
We reuse the declared triangle, Wheatstone, its published equal-reactance
comparison, and the saved System E specification. No graph, flow-noise, or market
samples are generated. System E's saved specification has rounded weights;
it is not silently substituted for the full-precision market fixture.

1. Label carriers, operators and their inner products.
2. Decompose the existing Wheatstone circulation example.
3. Reorient coordinates and verify unchanged physical results.
4. Relate resistance to a correctly weighted PTDF metric.
5. Reproduce the book's symmetry cancellation and asymmetric baseline.

In [ ]:
NOTEBOOK_ID='NB-04'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return response['result']
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

print('Local declared network inspection:',inputs['requests']['wheatstone']['inputs']['electrical_network'])
network_results={name:execute(name,'dc_network_response',request) for name,request in inputs['requests'].items()}
def model(name):
    bundle=network_results[name]['network_operator_bundle'];context=bundle['context']
    return dict(a=np.asarray(context['incidence']),w=np.diag(context['susceptance']),flow_metric=np.diag(1/np.asarray(context['susceptance'])),
        lap=np.asarray(bundle['laplacian']),lp=np.asarray(bundle['green']),psi=np.asarray(bundle['shift_factor_operator']))
models={name:model(name) for name in inputs['original_networks'] if name!='bridge_out'}
resistance_values={};resistance_pair_counts={}
for name in inputs['original_networks']:
    bundle=network_results[name]['network_operator_bundle'];network=bundle['context'];nodes=network['node_ids']
    context=dict(model_context_id=network['model_context_id'],topology_id=network['topology_state_id'],valid_support_id=network['valid_interval']['id'],
        weight_meaning='inverse_declared_reactance',weight_unit='normalized_susceptance',operator_unit='normalized_susceptance',inner_product='Euclidean',nullspace='one_constant_per_component')
    pairs=[[a,b] for a in nodes for b in nodes];values=[]
    # Partition the full ordered pair carrier to respect the existing 256-pair limit.
    for start in range(0,len(pairs),256):
        chosen=pairs[start:start+256];identifiers=[str(j) for j in range(start,start+len(chosen))]
        request=dict(inputs=dict(laplacian_operator=dict(node_ids=nodes,values=bundle['laplacian'],definition_id='construct.graph.combinatorial_laplacian.v1',context=context),
            node_pairs=dict(node_ids=nodes,pair_ids=identifiers,values=chosen,disconnected_policy='null',context=context)),parameters={},
            context=dict(input_origin='Complete original NB-04 ordered pair carrier; exact Laplacian from parent network execution; no geographic or market-capacity claim'))
        distance=execute(name+':resistance:'+str(start),'effective_resistance',request,[name])['resistance_distances']
        assert distance['pairs']==chosen and distance['pair_ids']==identifiers and not any(distance['disconnected_mask'])
        values.extend(distance['values'])
    assert len(values)==len(nodes)**2
    resistance_values[name]=np.asarray(values).reshape(len(nodes),len(nodes));resistance_pair_counts[name]=len(values)
for name,op in models.items():
    print(name,{'A_edges_by_buses':op['a'].shape,'W_edge_metric':op['w'].shape,'L_bus_operator':op['lap'].shape,'Psi_edges_by_buses':op['psi'].shape})
print('A: nodal potential → edge difference; W A: nodal potential → flow; Aᵀ: flow → nodal injection.')

## The metric determines the adjoint
For the nodal Euclidean inner product and flow metric W⁻¹,
⟨W A θ, f⟩_(W⁻¹) = ⟨θ, Aᵀ f⟩. The adjoint of **W A** is Aᵀ in these spaces.
The adjoint of **A** would instead be AᵀW⁻¹ if its codomain also used the flow
metric; do not swap these two statements. Edge differences use a different
natural metric W. The graph Laplacian is Aᵀ W A.

Flow projection is Q = W A L⁺ Aᵀ. It is idempotent and self-adjoint in W⁻¹,
though generally not symmetric in Euclidean coordinates. For a passive connected
graph, it picks the minimum-energy flow among those with the same divergence.

In [ ]:
adjoint_checks={}
for name,op in models.items():
    a,w,m=op['a'],op['w'],op['flow_metric'];q=op['psi']@a.T
    np.testing.assert_allclose((w@a).T@m,a.T,atol=1e-11)
    np.testing.assert_allclose(q@q,q,atol=1e-10)
    np.testing.assert_allclose(q.T@m,m@q,atol=1e-10)
    np.testing.assert_allclose(a.T@q,a.T,atol=1e-10)
    adjoint_checks[name]={'projection_residual':float(np.linalg.norm(q@q-q)),
        'weighted_self_adjoint_residual':float(np.linalg.norm(q.T@m-m@q)),
        'euclidean_asymmetry':float(np.linalg.norm(q.T-q))}
    print(name,adjoint_checks[name])

## The existing circulation experiment
Use the exact-identity test's 1 MW transfer 1→4 and its 0.7 MW circulation
around 1→2→3→1. In edge order (12,13,24,34,23), the cycle is (1,−1,0,0,1).
Adding it changes edge flows without changing any nodal injection. Projection
recovers the physical gradient flow; the cycle is orthogonal only in the
declared W⁻¹ inner product. Its extra energy is a quadratic proxy, not dollar cost.

In [ ]:
op=models['wheatstone'];a,w,m=op['a'],op['w'],op['flow_metric']
p=np.array([1.,0.,0.,-1.]);cycle=np.array([1.,-1.,0.,0.,1.])
physical=np.asarray(network_results['wheatstone']['flow_field']['values'][0]);alternative=physical+np.asarray(inputs['given_circulation'])
net=network_results['wheatstone']['network_operator_bundle']['context'];nodes=net['node_ids']
endpoints=[[nodes[int(np.flatnonzero(row==1)[0])],nodes[int(np.flatnonzero(row==-1)[0])]] for row in a]
# The existing Hodge profile uses a normalized base of 1 MVA, not a physical rating.
context=dict(model_context_id=net['model_context_id'],topology_state_id=net['topology_state_id'],valid_interval=net['valid_interval'],
    weight_unit='normalized_susceptance',weight_meaning='inverse_declared_reactance',flow_unit='MW',base_power_mva=1.,edge_orientation='positive_from_first_endpoint_to_second')
request=dict(inputs=dict(electrical_network=dict(node_ids=nodes,edge_ids=net['constraint_ids'],endpoints=endpoints,weights=net['susceptance'],active_mask=net['active_mask'],island_policy='reject',context=context),
    edge_flow_vector=dict(edge_ids=net['constraint_ids'],values=alternative.tolist(),context=context)),parameters={},
    context=dict(input_origin='Original NB-04 Wheatstone network and approved 0.7 MW circulation; normalized base, not an observed network rating'))
hodge=execute('wheatstone:hodge','weighted_hodge',request,['wheatstone'])['hodge_decomposition']
gradient=np.asarray(hodge['gradient']);circulation=np.asarray(hodge['cycle'])
np.testing.assert_allclose(a.T@cycle,0,atol=1e-12)
np.testing.assert_allclose(gradient,physical,atol=1e-12)
np.testing.assert_allclose(circulation,.7*cycle,atol=1e-12)
np.testing.assert_allclose(gradient@m@circulation,0,atol=1e-12)
energy=lambda f:float(f@m@f)
np.testing.assert_allclose(energy(alternative),energy(gradient)+energy(circulation),atol=1e-12)
print({'physical_flows':physical.tolist(),'weighted_inner_product':float(gradient@m@circulation),
       'euclidean_inner_product':float(gradient@circulation),
       'physical_energy':energy(physical),'circulation_energy':energy(circulation),'alternative_energy':energy(alternative)})
plt=setup_plot();fig,ax=plt.subplots(figsize=(10,4));x=np.arange(5)
for offset,values,label,color in [(-.25,alternative,'Given flow','#B48EAD'),(0,gradient,'Physical projection','#88C0D0'),(.25,circulation,'Circulation','#D08770')]:
    ax.bar(x+offset,values,.25,label=label,color=color)
ax.set_xticks(x,['1→2','1→3','2→4','3→4','2→3']);ax.axhline(0,color='#4C566A');ax.set(ylabel='MW',title='Same nodal injections, different edge flow');ax.legend();fig.tight_layout();plt.show()

## Orientation changes coordinates, not power flow
Reverse every declared edge coordinate: A′=−A, Ψ′=−Ψ and f′=−f. The Laplacian,
nodal injections, resistance, energy, and recovered physical flow after mapping
back are unchanged. Reordering or reversing only one operand would be an error.
This coordinate transformation is not a directed-affinity model or a topology event.

In [ ]:
orientation={}
for name,g in models.items():
    sign=-np.eye(len(g['a']));ar=sign@g['a'];wr=sign@g['w']@sign.T
    reversed_model=model(name+':reverse');lr=reversed_model['lap'];psir=reversed_model['psi']
    np.testing.assert_allclose(reversed_model['a'],ar);np.testing.assert_allclose(reversed_model['w'],wr)
    np.testing.assert_allclose(lr,g['lap'],atol=1e-10)
    np.testing.assert_allclose(sign.T@psir,g['psi'],atol=1e-10)
    orientation[name]=float(np.linalg.norm(sign.T@psir-g['psi']))
print('Mapped-back PTDF residuals:',orientation)

## Resistance equals the PTDF metric only with the right weights
For a balanced nodal transfer h, R(h)=hᵀL⁺h=(Ψh)ᵀW⁻¹(Ψh).
Equivalently ΨᵀW⁻¹Ψ=L⁺. Identity-weighted constraint exposure in NB-15 has a
different interpretation; it does not generally equal effective resistance.
The Wheatstone endpoint resistance is 1.4 with the bridge and 1.5 without it.
Removing the bridge leaves this example connected; no islanding convention is
silently inferred from a pseudoinverse.

In [ ]:
resistance_checks={}
for name,g in models.items():
    metric=g['psi'].T@g['flow_metric']@g['psi']
    np.testing.assert_allclose(metric,g['lp'],atol=1e-10)
    r=resistance_values[name];np.testing.assert_allclose(np.diag(r),0,atol=1e-10)
    assert r.min()>-1e-10
    resistance_checks[name]=float(np.linalg.norm(metric-g['lp']))

# Existing Model C bridge-outage case, with the first four declared edges.
bridge_out=model('bridge_out')
np.testing.assert_allclose([resistance_values['wheatstone'][0,3],resistance_values['bridge_out'][0,3]],[1.4,1.5],atol=1e-12)
print('Resistance identity residuals:',resistance_checks)
print('Wheatstone 1→4: resistance',p@op['lp']@p,'; unweighted PTDF norm squared',physical@physical)
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,name in zip(axes,('wheatstone','system_e_spec')):
    r=resistance_values[name];im=ax.imshow(r,cmap='Blues');fig.colorbar(im,ax=ax,label='Resistance in declared weight units')
    ax.set(title=name+' · effective resistance',xlabel='Bus index',ylabel='Bus index')
    if name=='wheatstone':
        ax.set_xticks(range(4),[1,2,3,4]);ax.set_yticks(range(4),[1,2,3,4]);ax.set(xlabel='Published bus label',ylabel='Published bus label')
fig.tight_layout();plt.show()

## Wheatstone cancellation is specific to an edge–transfer pair
The book explicitly compares all-equal reactances against its asymmetric
baseline (1,2,2,1,1). With equal reactances the 1→4 transfer cancels on the bridge;
the same bridge still responds to 1→2. Returning to the published asymmetric
weights breaks that cancellation and gives 0.2 MW bridge flow. These are the
existing two cases, not a new interpolated scenario sweep.

In [ ]:
symmetric=models['symmetric_wheatstone'];h12=np.array([1.,-1.,0.,0.])
flows=np.column_stack([network_results[name]['flow_field']['values'][0] for name in ('symmetric_wheatstone','symmetric_wheatstone:h12','wheatstone')])
np.testing.assert_allclose(flows[4,0],0,atol=1e-12)
assert abs(flows[4,1])>1e-3
np.testing.assert_allclose(flows[:,2],[.6,.4,.4,.6,.2],atol=1e-12)
print('Bridge responses:',flows[4].tolist())
fig,ax=plt.subplots(figsize=(10,4))
im=ax.imshow(flows.T,aspect='auto',cmap='RdBu',vmin=-abs(flows).max(),vmax=abs(flows).max())
ax.set_xticks(range(5),['1→2','1→3','2→4','3→4','2→3'])
ax.set_yticks(range(3),['Equal weights · 1→4','Equal weights · 1→2','Published baseline · 1→4'])
ax.set(title='A zero response does not make an edge irrelevant');fig.colorbar(im,ax=ax,label='MW per 1 MW transfer');fig.tight_layout();plt.show()

## Exercise and boundaries
Why can the Euclidean inner product between physical and cycle flow be nonzero
while the weighted product vanishes? The coordinates are flow, so their energy
metric is W⁻¹. Recompute with the wrong metric and compare above.

These identities describe a passive DC network without phase-control offsets.
A physical phase shifter can introduce an affine circulation; the injection
derivative and baseline flow then require separate treatment. No AC or market
price claim follows from this graph-only calculation. The System E display uses
the existing rounded JSON weights and does not rerun the seeded graph generator.

In [ ]:
bad=copy.deepcopy(inputs['requests']['wheatstone']);bad['inputs']['balanced_injections']['node_ids'].reverse();reject('dc_network_response',bad)
bad=copy.deepcopy(inputs['requests']['wheatstone']);bad['inputs']['electrical_network']['susceptance'][0]=0.;reject('dc_network_response',bad)
def plain(value):
    if isinstance(value,np.ndarray):return value.tolist()
    raise TypeError(type(value))
summary_data=dict(adjoints=adjoint_checks,orientation=orientation,resistance_checks=resistance_checks,resistance=resistance_values,
    resistance_pair_counts=resistance_pair_counts,models=models,flow_decomposition=dict(given=alternative,physical=gradient,cycle=circulation),
    hodge=hodge,Wheatstone_comparison=flows)
(session.output/'consumer-summary.json').write_text(json.dumps(summary_data,default=plain,allow_nan=False,indent=2)+'\n')
summary=session.finish(dict(weighted_adjoint=True,Hodge_projection=True,orientation_invariance=True,resistance_metric_identity=True,
    complete_676_System_E_pairs=True,published_Wheatstone_cases=True,new_synthetic_data=False),dict(system_e_precision=inputs['source_precision']))
print(summary)